# Transformer Translation Baseline

目标：复现仓库中的编码器—解码器 Transformer 翻译基线，验证核心模块、训练流程、验证指标和生成结果。

成功标准：

- 缩放点积注意力、mask、shape 和批量贪心解码检查通过。
- 训练脚本完成 20 个 epoch 并保存模型、曲线和验证结果。
- 能定位最佳验证 epoch，并说明训练 loss 持续下降而验证 loss 回升的含义。


## 1. 设置与可复现性

Notebook 可以从仓库根目录或 `experiments` 目录启动。模型与训练脚本内部使用固定随机种子 `42`，本基线显式使用 CPU 以避免不同 CUDA 环境改变结果。


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "transformer_translation.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('C:/Users/20571/Desktop/machine_learing')

## 2. 实验计划

1. 运行 Transformer 核心模块检查。
2. 使用固定数据切分和随机种子重新训练。
3. 读取逐 epoch 的训练 loss、验证 loss 和 token accuracy。
4. 比较最佳验证 epoch 与最后一个 epoch。
5. 检查生成样例并记录实验边界。


In [2]:
import runpy

runpy.run_path(
    str(PROJECT_ROOT / "transformer_translation.py"),
    run_name="__main__",
)


Scaled dot-product attention checks passed.


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'C:\\Users\\20571\\Desktop\\machine_learing\\transformer_translation.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, loc

## 3. 训练基线

训练数据为 10 条中英文本对，验证数据为 2 条。该实验只验证完整流程，不代表有效翻译质量。


In [3]:
from train_transformer import main as train_transformer

train_transformer(device="cpu")


epoch=01 train_loss=4.8233 val_loss=4.7097
epoch=02 train_loss=4.5511 val_loss=4.6398
epoch=03 train_loss=4.3646 val_loss=4.6005
epoch=04 train_loss=4.1876 val_loss=4.5767
epoch=05 train_loss=4.0343 val_loss=4.5734
epoch=06 train_loss=3.8756 val_loss=4.5838
epoch=07 train_loss=3.7239 val_loss=4.6087
epoch=08 train_loss=3.5713 val_loss=4.6316
epoch=09 train_loss=3.4279 val_loss=4.6421
epoch=10 train_loss=3.2727 val_loss=4.6445
epoch=11 train_loss=3.1383 val_loss=4.6536


epoch=12 train_loss=2.9953 val_loss=4.6762
epoch=13 train_loss=2.8570 val_loss=4.6992
epoch=14 train_loss=2.7079 val_loss=4.7086
epoch=15 train_loss=2.5724 val_loss=4.7150
epoch=16 train_loss=2.4740 val_loss=4.7294
epoch=17 train_loss=2.3266 val_loss=4.7445
epoch=18 train_loss=2.2000 val_loss=4.7528
epoch=19 train_loss=2.0704 val_loss=4.7619
epoch=20 train_loss=1.9668 val_loss=4.7680
saved=C:\Users\20571\Desktop\machine_learing\outputs


## 4. 读取结果


In [4]:
RESULT_PATH = PROJECT_ROOT / "outputs" / "validation_results.json"
results = json.loads(RESULT_PATH.read_text(encoding="utf-8"))

{
    "seed": results["seed"],
    "train_examples": results["train_examples"],
    "validation_examples": results["validation_examples"],
    "best_validation": results["best_validation"],
    "final_train": results["final_train"],
}


{'seed': 42,
 'train_examples': 10,
 'validation_examples': 2,
 'best_validation': {'epoch': 5,
  'train_loss': 4.034274697303772,
  'train_token_accuracy': 0.18226600985221675,
  'val_loss': 4.573384761810303,
  'val_token_accuracy': 0.10256410256410256},
 'final_train': {'epoch': 20,
  'train_loss': 1.9667767882347107,
  'train_token_accuracy': 0.729064039408867,
  'val_loss': 4.768017768859863,
  'val_token_accuracy': 0.15384615384615385}}

## 5. 过拟合检查

最佳验证 loss 与最后一个 epoch 的验证 loss 应分开比较。训练 loss 继续下降而验证 loss 上升，表示模型在微型训练集上继续拟合，但泛化没有继续改善。


In [5]:
best = results["best_validation"]
final = results["history"][-1]
analysis = {
    "best_epoch": best["epoch"],
    "best_val_loss": best["val_loss"],
    "final_epoch": final["epoch"],
    "final_train_loss": final["train_loss"],
    "final_val_loss": final["val_loss"],
    "validation_loss_increased_after_best": (
        final["val_loss"] > best["val_loss"]
    ),
}
analysis


{'best_epoch': 5,
 'best_val_loss': 4.573384761810303,
 'final_epoch': 20,
 'final_train_loss': 1.9667767882347107,
 'final_val_loss': 4.768017768859863,
 'validation_loss_increased_after_best': True}

## 6. 生成样例


In [6]:
{
    "generated": results["generated_validation_text"],
    "reference": results["reference_validation_text"],
}


{'generated': '我了，，，，，，，，，，，，，，，，，，，，，，，，，，，，，',
 'reference': '我<UNK>了<UNK>长时间，<UNK><UNK>得他<UNK><UNK><UNK><UNK><UNK>来。'}

## 7. 结论与边界

- 模型、训练、验证、保存和生成流程能够端到端复现。
- 最佳验证 loss 出现在第 5 个 epoch，之后训练 loss 继续下降而验证 loss 回升。
- 数据量只有 12 条，生成样例不能用于判断真实翻译质量。
- 后续扩大实验时，应增加数据量、使用更稳定的文本级指标，并报告多个随机种子。
